# Model Comparison and Feature Selection

**Autor** : Vukasin Jovanovic

**Datum** : 27.08.2026

## Cilj Analize
Cilj ove analize je uporediti različite regresione modele i ispitati uticaj različitih kombinacija karakteristika na performanse modela, kako bi se odabrala najbolja kombinacija modela i karakteristika za predikciju cene automobila.

## Uvoz potrebnih biblioteka

In [9]:
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import sys
from pathlib import Path
from sklearn.compose import ColumnTransformer

sys.path.append(str(Path.cwd().parent / "src"))

from data_preprocessing import (
    split_features_and_target,
    build_preprocessor,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    _build_numeric_transformer,
    _build_categorical_transformer,
)

## Ucitavanje podataka

In [3]:
DATA_PATH = "../data/cars_cleaned_with_features.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,make,model,price_usd,year,condition,mileage_kilometers,fuel_type,volume_cm3,color,transmission,drive_unit,segment,car_age,mileage_per_year,brand_model
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,b,18,9000.000000,mazda_2
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,b,17,7058.823529,mazda_2
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,b,17,3588.235294,mazda_2
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,b,23,11521.739130,mazda_2
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,b,18,5399.055556,mazda_2


## Podela na trening/test skupove podataka

In [4]:
X, y = split_features_and_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## All Features

In [5]:
feature_sets = {
    "All Features": X.columns.tolist()
}

feature_sets

{'All Features': ['year',
  'mileage_kilometers',
  'volume_cm3',
  'car_age',
  'mileage_per_year',
  'make',
  'model',
  'condition',
  'fuel_type',
  'color',
  'transmission',
  'drive_unit',
  'segment',
  'brand_model']}

## Models

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

## Baseline — All Features

In [7]:
results = []

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "All Features",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
1,All Features,Ridge Regression,2109.379057,1.874642e+07,4329.714061,0.752500
0,All Features,Linear Regression,2115.558468,1.895024e+07,4353.187436,0.749809


## Without `brand_model`

In [8]:
feature_sets["Without brand_model"] = [
    feature for feature in X.columns
    if feature != "brand_model"
]

print(feature_sets["Without brand_model"])

['year', 'mileage_kilometers', 'volume_cm3', 'car_age', 'mileage_per_year', 'make', 'model', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment']


In [10]:
def build_experiment_preprocessor(features):
    numeric_features = [
        feature for feature in features
        if feature in NUMERIC_FEATURES
    ]

    categorical_features = [
        feature for feature in features
        if feature in CATEGORICAL_FEATURES
    ]

    return ColumnTransformer(
        transformers=[
            ("num", _build_numeric_transformer(), numeric_features),
            ("cat", _build_categorical_transformer(), categorical_features),
        ],
        remainder="drop"
    )

In [11]:
features = feature_sets["Without brand_model"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without brand_model",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869
1,All Features,Ridge Regression,2109.379057,1.874642e+07,4329.714061,0.752500
5,Without brand_model,Ridge Regression,2114.008156,1.906794e+07,4366.685543,0.748255
0,All Features,Linear Regression,2115.558468,1.895024e+07,4353.187436,0.749809
4,Without brand_model,Linear Regression,2118.196601,1.933784e+07,4397.481308,0.744692


`brand_model` ne poboljšava performanse Random Forest modela i njegovo uklanjanje daje blago bolje rezultate.

## Without `make` and `model`

In [12]:
feature_sets["Without make and model"] = [
    feature for feature in X.columns
    if feature not in ["make", "model"]
]

print(feature_sets["Without make and model"])

['year', 'mileage_kilometers', 'volume_cm3', 'car_age', 'mileage_per_year', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment', 'brand_model']


In [13]:
features = feature_sets["Without make and model"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without make and model",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869
11,Without make and model,Gradient Boosting,1643.812624,1.006230e+07,3172.113645,0.867152
1,All Features,Ridge Regression,2109.379057,1.874642e+07,4329.714061,0.752500
5,Without brand_model,Ridge Regression,2114.008156,1.906794e+07,4366.685543,0.748255
0,All Features,Linear Regression,2115.558468,1.895024e+07,4353.187436,0.749809
4,Without brand_model,Linear Regression,2118.196601,1.933784e+07,4397.481308,0.744692


## Without `volume_cm3`

In [14]:
feature_sets["Without volume"] = [
    feature for feature in X.columns
    if feature != "volume_cm3"
]

In [15]:
features = feature_sets["Without volume"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without volume",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
14,Without volume,Random Forest,1181.935129,9.486632e+06,3080.037734,0.874753
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869
11,Without make and model,Gradient Boosting,1643.812624,1.006230e+07,3172.113645,0.867152
15,Without volume,Gradient Boosting,1806.501036,1.355613e+07,3681.864715,0.821025
1,All Features,Ridge Regression,2109.379057,1.874642e+07,4329.714061,0.752500
5,Without brand_model,Ridge Regression,2114.008156,1.906794e+07,4366.685543,0.748255


## Without `car_age`

In [16]:
feature_sets["Without car_age"] = [
    feature for feature in X.columns
    if feature != "car_age"
]

In [17]:
features = feature_sets["Without car_age"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without car_age",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
18,Without car_age,Random Forest,1079.821285,6.786574e+06,2605.105450,0.910400
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
14,Without volume,Random Forest,1181.935129,9.486632e+06,3080.037734,0.874753
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
19,Without car_age,Gradient Boosting,1557.667517,8.898109e+06,2982.969880,0.882523
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869
11,Without make and model,Gradient Boosting,1643.812624,1.006230e+07,3172.113645,0.867152
15,Without volume,Gradient Boosting,1806.501036,1.355613e+07,3681.864715,0.821025


## Without `mileage_per_year`

In [18]:
feature_sets["Without mileage_per_year"] = [
    feature for feature in X.columns
    if feature != "mileage_per_year"
]

In [19]:
features = feature_sets["Without mileage_per_year"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without mileage_per_year",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
18,Without car_age,Random Forest,1079.821285,6.786574e+06,2605.105450,0.910400
22,Without mileage_per_year,Random Forest,1079.987174,6.752706e+06,2598.596961,0.910847
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
14,Without volume,Random Forest,1181.935129,9.486632e+06,3080.037734,0.874753
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
19,Without car_age,Gradient Boosting,1557.667517,8.898109e+06,2982.969880,0.882523
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869
23,Without mileage_per_year,Gradient Boosting,1567.118885,9.015673e+06,3002.610978,0.880971


## Without `brand_model` + `car_age`

In [20]:
feature_sets["Without brand_model and car_age"] = [
    feature for feature in X.columns
    if feature not in ["brand_model", "car_age"]
]

In [21]:
features = feature_sets["Without brand_model and car_age"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without brand_model and car_age",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
26,Without brand_model and car_age,Random Forest,1078.935321,6.713292e+06,2591.002128,0.911368
18,Without car_age,Random Forest,1079.821285,6.786574e+06,2605.105450,0.910400
22,Without mileage_per_year,Random Forest,1079.987174,6.752706e+06,2598.596961,0.910847
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
14,Without volume,Random Forest,1181.935129,9.486632e+06,3080.037734,0.874753
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
19,Without car_age,Gradient Boosting,1557.667517,8.898109e+06,2982.969880,0.882523
7,Without brand_model,Gradient Boosting,1558.661659,8.871873e+06,2978.568936,0.882869


## Without `brand_model` + `mileage_per_year`

In [22]:
feature_sets["Without brand_model and mileage_per_year"] = [
    feature for feature in X.columns
    if feature not in ["brand_model", "mileage_per_year"]
]

In [23]:
features = feature_sets["Without brand_model and mileage_per_year"]

for model_name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", build_experiment_preprocessor(features)),
            ("regressor", regressor),
        ]
    )

    model.fit(X_train[features], y_train)

    y_pred = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, y_pred)

    results.append({
        "feature_set": "Without brand_model and mileage_per_year",
        "model": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    })

results_df = pd.DataFrame(results)

results_df.sort_values("mae")

,feature_set,model,mae,mse,rmse,r2
6,Without brand_model,Random Forest,1078.184231,6.711611e+06,2590.677754,0.911390
30,Without brand_model and mileage_per_year,Random Forest,1078.600252,6.719474e+06,2592.194750,0.911286
26,Without brand_model and car_age,Random Forest,1078.935321,6.713292e+06,2591.002128,0.911368
18,Without car_age,Random Forest,1079.821285,6.786574e+06,2605.105450,0.910400
22,Without mileage_per_year,Random Forest,1079.987174,6.752706e+06,2598.596961,0.910847
2,All Features,Random Forest,1081.625954,6.808465e+06,2609.303518,0.910111
10,Without make and model,Random Forest,1116.827885,7.484077e+06,2735.704130,0.901191
14,Without volume,Random Forest,1181.935129,9.486632e+06,3080.037734,0.874753
3,All Features,Gradient Boosting,1557.422385,8.864760e+06,2977.374760,0.882963
19,Without car_age,Gradient Boosting,1557.667517,8.898109e+06,2982.969880,0.882523


## Zakljucak

Na osnovu sprovedenih eksperimenata izabran je **Random Forest Regressor** kao najbolji model za predikciju cene automobila. Od svih testiranih modela, Random Forest je ostvario najbolje rezultate po svim glavnim metrikama: najniži MAE i RMSE, kao i najviši R² rezultat.

Kao najbolja kombinacija karakteristika izabrana je kombinacija **bez karakteristike `brand_model`**, dok se sve ostale karakteristike zadržavaju. Ova kombinacija je ostvarila:

- **MAE:** 1,078.18 $
- **RMSE:** 2,590.68 $
- **R²:** 0.9114

Karakteristika `brand_model` je izostavljena jer je predstavlja kombinaciju već postojećih karakteristika `make` i `model`, pa nije donela dodatnu korisnu informaciju modelu. Štaviše, uklanjanjem `brand_model` dobijen je blago bolji rezultat nego korišćenjem svih karakteristika.

Ostale karakteristike su zadržane. Posebno je potvrđeno da `volume_cm3` ima značajan doprinos, jer je njegovo uklanjanje dovelo do primetnog pogoršanja performansi. Takođe su zadržane `make` i `model`, jer njihovo uklanjanje takođe dovodi do pogoršanja rezultata. `car_age` i `mileage_per_year` su zadržane jer njihovo uklanjanje nije donelo poboljšanje u odnosu na najbolju kombinaciju.

Konačan izbor za dalji razvoj modela je, dakle, **Random Forest Regressor sa svim originalnim i izvedenim karakteristikama osim `brand_model`**. Ovaj model predstavlja osnovu za narednu fazu projekta, u kojoj će se izvršiti podešavanje hiperparametara sa ciljem dodatnog poboljšanja performansi.